# 워밍업 — 2점 차이의 진실 🎯

웹 챕터에서 만진 실험을 이번엔 **직접 코드로** 돌려 봅니다 — 같은 난수열(시드 고정)이라 웹과 숫자가 정확히 같습니다.

> **노트북 사용법**: 각 셀(회색 상자)을 클릭하고 `Shift + Enter` — 위에서 아래로 반복하면 끝.
> 코드를 다 이해하지 못해도 괜찮아요. 실행해서 결과를 눈으로 보는 게 목적입니다.

**이 노트북의 목표**: 10부 전체의 출발 질문을 코드로 확인한다 —
**"벤치마크 점수는 실력이 아니라 측정이다 — 문항 수가 적으면 순위는 동전 던지기가 된다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('sample')

## 1. 무대 — 신의 시점으로 실험 설계

현실에서는 모델의 "진짜 실력"을 아무도 모릅니다 — 시험(벤치마크)으로 추정할 뿐이죠.
그래서 실험실에서만 가능한 반칙을 씁니다: **진짜 실력을 우리가 미리 정해 두는 것.**

- 모델 A — 진짜 실력 **72%** (문제 하나를 맞힐 확률)
- 모델 B — 진짜 실력 **70%** → A가 2%p 더 잘함

이제 문항 수 n짜리 시험을 치게 하고, **시험 점수가 이 순서를 맞히는지** 봅니다.
난수는 mulberry32(시드 = n×100 + 회차)로 고정 — 웹 랩과 **비트 단위로 같은** 난수열입니다.

In [ ]:
M = 0xFFFFFFFF  # 32비트 마스크 — JS 의 32비트 정수 연산을 그대로 흉내 낸다

def imul(a, b):
    """JS Math.imul — 32비트 곱셈의 하위 32비트."""
    return (a * b) & M

def mulberry32(seed):
    """웹 랩(ScoreFlipLab.vue)과 동일한 난수 생성기 — 같은 시드면 같은 난수열."""
    a = seed & M
    def rand():
        nonlocal a
        a = (a + 0x6D2B79F5) & M
        t = a
        t = imul(t ^ (t >> 15), t | 1) & M
        t = (t ^ ((t + imul(t ^ (t >> 7), t | 61)) & M)) & M
        return (t ^ (t >> 14)) / 4294967296
    return rand

# 웹(JS)에서 뽑은 검증값과 비트 단위로 일치하는지 확인 — 포트가 정확하다는 증거
r = mulberry32(1)
vals = [r() for _ in range(3)]
expect = [0.627073940588, 0.002735721180, 0.527447039960]
for v, e in zip(vals, expect):
    assert abs(v - e) < 1e-12, (v, e)
print('✔ mulberry32 포트 검증 — JS(웹 랩)와 같은 난수열:', ', '.join(f'{v:.6f}' for v in vals))

PA, PB = 0.72, 0.70  # 신의 시점 — 진짜 실력

## 2. 실험 — 문항 수별로 시험을 20번씩 다시 내 보기

시험 1회 = 문항 n개를 각각 확률(실력)대로 맞히거나 틀리는 것.
같은 문항 수로 20번 다시 채점하면서, **실력이 낮은 B가 이기는 시험이 몇 번인지** 셉니다.

In [ ]:
def scores(n, rnd):
    """n문항 시험 1회 — (A 점수, B 점수). 결정적: 같은 (n, 회차)는 언제나 같은 점수."""
    rand = mulberry32(n * 100 + rnd)
    a = sum(1 for _ in range(n) if rand() < PA)
    b = sum(1 for _ in range(n) if rand() < PB)
    return a / n * 100, b / n * 100

NS = [25, 50, 100, 250, 1000, 5000]
ROUNDS = 20

print('문항 수 |  B승(뒤집힘)  동점 | 1회차 점수 (A vs B)')
print('-' * 58)
table = {}
for n in NS:
    flip = tie = 0
    for rnd in range(1, ROUNDS + 1):
        sa, sb = scores(n, rnd)
        if sb > sa:
            flip += 1
        elif sb == sa:
            tie += 1
    sa1, sb1 = scores(n, 1)
    table[n] = (flip, tie)
    mark = ' ← 동전 던지기!' if flip >= 8 else (' ← 이제 안정' if flip == 0 else '')
    print(f'{n:>6} |     {flip:>2}번      {tie:>2}번 | A {sa1:5.1f} vs B {sb1:5.1f}{mark}')

# 웹 챕터 본문에 적힌 수치와 정확히 일치하는지 못박기 (콘텐츠-코드 불일치 방지)
assert table[25] == (6, 0)
assert table[50] == (10, 1), '50문항: B승 10번 + 동점 1번 — 본문의 핵심 수치'
assert table[100] == (8, 0)
assert table[250] == (5, 2)
assert table[1000] == (0, 0), '1,000문항부터 뒤집힘 0'
assert table[5000] == (0, 0)
assert scores(50, 1) == (70.0, 76.0), '웹 랩 기본 화면(50문항 1회차) — B가 76:70으로 역전'
print()
print('✔ 전 수치가 웹 챕터 본문과 일치 — 진짜 실력 A>B 인데 50문항에선 절반이 뒤집혔다')

## 3. 그림으로 — 뒤집힘이 사라지는 순간

왼쪽: 문항 수별 "B가 이긴 시험 수". 오른쪽: 1회차 점수에 **±오차범위**를 씌운 모습 —
50문항의 밴드는 두 모델이 크게 겹치지만, 5,000문항이면 바짝 좁아집니다.

In [ ]:
import matplotlib.pyplot as plt
import math

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'  # 한글 라벨 (Windows)
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

band = lambda p100, n: 1.96 * math.sqrt((p100 / 100) * (1 - p100 / 100) / n) * 100  # ±폭(%p)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

# (좌) 문항 수별 뒤집힘
labels = [str(n) for n in NS]
flips = [table[n][0] for n in NS]
ax1.bar(labels, flips, color='#fb7185')
ax1.axhline(10, color='#94a3b8', ls='--', lw=1)
ax1.text(5.4, 10.3, '동전 던지기(10/20)', ha='right', fontsize=9, color='#64748b')
ax1.set_title('실력이 낮은 B가 이긴 시험 수 (20회 중)')
ax1.set_xlabel('문항 수')
ax1.set_ylim(0, 12)

# (우) 1회차 점수 ± 오차범위
for i, n in enumerate([50, 5000]):
    sa, sb = scores(n, 1)
    ax2.errorbar([i * 2], [sa], yerr=[band(sa, n)], fmt='o', color='#6366f1', capsize=6, ms=8)
    ax2.errorbar([i * 2 + 0.6], [sb], yerr=[band(sb, n)], fmt='o', color='#fb7185', capsize=6, ms=8)
ax2.set_xticks([0.3, 2.3])
ax2.set_xticklabels(['50문항\n(밴드가 크게 겹침)', '5,000문항\n(바짝 좁아짐)'])
ax2.axhline(72, color='#6366f1', ls=':', lw=1)
ax2.axhline(70, color='#fb7185', ls=':', lw=1)
ax2.set_title('1회차 점수 ± 오차범위 (점선 = 진짜 실력)')
ax2.set_ylabel('점수')
plt.tight_layout()
plt.show()
print('파랑 = A(실력 72), 빨강 = B(실력 70)')

## 4. 오차범위 맛보기 — 여론조사의 그 "±3.1%p"

점수의 출렁임 폭(±)은 공식 하나로 어림됩니다 — **± ≈ 1.96 × √(p(1−p)/n)**.
지금은 "문항 수 n이 분모에 있다 → n이 클수록 좁아진다"만 가져가면 충분합니다
(이 공식의 정체와 유도는 **6장**의 주인공 — 3부 8장의 신뢰구간이 그대로 회수됩니다).

재미있는 성질: **√n 꼴**이라 문항을 4배로 늘려야 ±가 절반이 됩니다 —
3부 7장(큰 수의 법칙)에서 본 그 속도예요.

In [ ]:
print('문항 수 | ±오차범위(p=0.70 기준)')
print('-' * 34)
for n in NS:
    print(f'{n:>6} |  ±{band(70, n):4.1f}%p')

assert round(band(70, 50), 1) == 12.7, '50문항 ±12.7%p — 재려는 차이(2%p)의 6배!'
assert round(band(70, 5000), 1) == 1.3, '5,000문항 ±1.3%p'
assert abs(band(70, 100) / band(70, 400) - 2) < 0.01, '문항 4배 → 오차범위 절반 (√n)'
print()
print('✔ 여론조사(n=1,000)의 ±3.1%p 도 같은 공식: ±', round(band(50, 1000), 1), '%p (p=0.5 기준)')

## 정리 — 오늘 가져갈 세 문장

1. **점수 = 실력 + 소음(표본 오차)** — 벤치마크 점수는 측정값이지 실력 그 자체가 아니다.
2. **소음의 크기는 문항 수가 정한다** — 50문항의 2점 차이는 동전 던지기, 1,000문항이면 믿을 만.
3. 실전 습관: 리더보드 점수를 보면 **문항 수부터 확인**한다.

다음 **1장 — 큰 그림**에서는 질문을 반대쪽에서 시작합니다:
"모델을 키우면 좋아진다는데, **얼마나**?" — 훈련의 3재료(파라미터 N·데이터 D·컴퓨트 C) 지도를 폅니다.